# Follow-Up Detection

Classify each radiology report for whether it recommends patient-specific follow-up, using an LLM served by Ollama.

**Pipeline:**
1. One-time copy of `default.reports_latest` into a working table `default.reports_followup`. The working table is owned by this notebook so concurrent hl7-transformer ingests don't fight our writes. The schema also carries the demographics, exam, and diagnosis columns that `followup_review_dashboard.py` (the Voilà playbook) reads when building its review UI.
2. Classify reports in batches via Ollama, with a priority filter that processes likely-positive reports (keyword match + complex modality) first.
3. MERGE results back into the working table. Errors go to `default.followup_errors`.

Re-run the top-up cell whenever new HL7 ingests have landed to bring the new accessions into the working table; then re-run the pipeline.

> ⚠️ **Don't "Run All".** Cell 5 is destructive — it drops `default.reports_followup`, wiping any classifier output and reviewer annotations. Run the notebook piecemeal:
> - **First time:** cells 1–3 → 5 (create working table) → 7–9 → 10–11 (test run) → 12–13 (full pipeline)
> - **After new HL7 ingests:** cells 1–3 → **6** (top-up, non-destructive) → 12–13 (classifier resumes from where it stopped)
> - **Anytime:** cell 15 (summary)


In [ ]:
import os
import json
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from tqdm.notebook import tqdm

In [ ]:
# --- Ollama ---
# install-chat (scout-demo) deploys Ollama at ollama.<chatbot_namespace>:11434.
# chatbot_namespace defaults to scout-analytics (see scout_common/defaults/main.yaml).
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://ollama.scout-analytics:11434")
MODEL = os.getenv("OLLAMA_MODEL", "gemma4-31b-long:latest")

# Match OLLAMA_NUM_PARALLEL on the deployed Ollama. install-chat doesn't set it,
# so Ollama auto-tunes (~4 on an 80GB A100). For a 31B-long model, 4 is a safe start;
# bump only after benchmarking and confirming GPU memory headroom.
TOTAL_WORKERS = 4
BATCH_SIZE = 1000

# --- S3 (MinIO) ---
S3_ACCESS_KEY = os.getenv("S3_ACCESS_KEY", "lake-writer")
S3_SECRET_KEY = os.getenv("S3_SECRET_KEY") or getpass("Enter S3 secret key: ")

# --- Tables ---
# hl7-transformer's deduped table (one row per accession_number, latest message_dt).
SRC_TABLE = "default.reports_latest"
# Working table owned by this notebook; copied once from SRC_TABLE.
WORK_TABLE = "default.reports_followup"
ERROR_TABLE = "default.followup_errors"

# --- Priority filter (process most-likely follow-ups first) ---
FOLLOWUP_KEYWORDS = [
    r"follow.?up", r"repeat", r"recommend", r"suggest", r"interval",
    r"re-?evaluate", r"re-?assess", r"correlat", r"clinical", r"short.?term",
    r"further", r"additional", r"consider", r"refer", r"return",
    r"continue", r"monitor", r"surveillance", r"re-?exam", r"comparison",
]
PRIORITY_MODALITIES = ["CT", "MR", "US", "MG", "PT"]
PRIORITY_FILTER = (
    f"(lower(report_text) RLIKE '{('|'.join(FOLLOWUP_KEYWORDS))}') AND "
    f"(modality IN ({','.join(repr(m) for m in PRIORITY_MODALITIES)}))"
)

print(f"Ollama:  {OLLAMA_URL}  ({MODEL})")
print(f"Workers: {TOTAL_WORKERS}, batch: {BATCH_SIZE:,}")
print(f"Source:  {SRC_TABLE}")
print(f"Work:    {WORK_TABLE}")
print(f"Errors:  {ERROR_TABLE}")

In [ ]:
# Hive metastore in scout-demo lives in the scout-data namespace.
# Use the WRITABLE service (not -readonly) since we create/append to follow-up tables.
spark = (
    SparkSession.builder.appName("followup-detection")
    .config("spark.hadoop.fs.s3a.access.key", S3_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", S3_SECRET_KEY)
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore.scout-data:9083")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "800")
    .enableHiveSupport()
    .getOrCreate()
)
spark.sql("SHOW DATABASES").show()
print("Spark connected to Delta Lake")

## One-time setup: working table

> ⚠️ **Destructive — run only on a fresh deployment.** Drops `default.reports_followup` and recreates it from `reports_latest`. Any prior classifier output and reviewer annotations are lost. After the first run, use the **Top-up** cell below to fold in new accessions without disturbing existing rows.

The fresh schema includes:

- **Key:** `primary_report_identifier` (used for MERGE) and `accession_number` (used by the Voilà playbook).
- **Classifier inputs:** `report_text`, `modality`.
- **Playbook display columns:** `service_name`, `service_identifier`, `message_dt`, `patient_age`, `sex`, `race`, `sending_facility`, `diagnoses`, `principal_result_interpreter`.
- **Classifier outputs:** `followup_detected`, `followup_confidence`, `followup_snippet`, `followup_finding`, `followup_processed_at`.

The cell below tops up new accessions on subsequent runs.


In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {WORK_TABLE}")
print(f"Creating {WORK_TABLE} from {SRC_TABLE} ...")
(
    spark.table(SRC_TABLE)
    .select(
        "primary_report_identifier",
        "accession_number",
        "report_text",
        "modality",
        "service_name",
        "service_identifier",
        "message_dt",
        "patient_age",
        "sex",
        "race",
        "sending_facility",
        "diagnoses",
        "principal_result_interpreter",
    )
    .withColumn("followup_detected", F.lit(None).cast("boolean"))
    .withColumn("followup_confidence", F.lit(None).cast("string"))
    .withColumn("followup_snippet", F.lit(None).cast("string"))
    .withColumn("followup_finding", F.lit(None).cast("string"))
    .withColumn("followup_processed_at", F.lit(None).cast("timestamp"))
    .write.format("delta")
    .saveAsTable(WORK_TABLE)
)
print(f"Created. Rows: {spark.table(WORK_TABLE).count():,}")

In [ ]:
# Top up: insert any rows in SRC_TABLE that aren't yet in WORK_TABLE.
# Run this whenever new HL7 ingests have landed and you want to classify them.
spark.sql(f"""
    INSERT INTO {WORK_TABLE}
    SELECT
        s.primary_report_identifier,
        s.accession_number,
        s.report_text,
        s.modality,
        s.service_name,
        s.service_identifier,
        s.message_dt,
        s.patient_age,
        s.sex,
        s.race,
        s.sending_facility,
        s.diagnoses,
        s.principal_result_interpreter,
        CAST(NULL AS BOOLEAN)   AS followup_detected,
        CAST(NULL AS STRING)    AS followup_confidence,
        CAST(NULL AS STRING)    AS followup_snippet,
        CAST(NULL AS STRING)    AS followup_finding,
        CAST(NULL AS TIMESTAMP) AS followup_processed_at
    FROM {SRC_TABLE} s
    LEFT ANTI JOIN {WORK_TABLE} w
        ON w.primary_report_identifier = s.primary_report_identifier
""")
print(f"{WORK_TABLE} rows: {spark.table(WORK_TABLE).count():,}")

## Classifier

Each classification returns four pieces:
- `follow_up` — whether the report recommends patient-specific follow-up
- `confidence` — `high` or `low`
- `finding` — the underlying clinical finding driving the recommendation (e.g., `"8 mm pulmonary nodule"`). Empty when `follow_up=false`.
- `snippet` — short verbatim excerpt from the report text containing the recommendation. Empty when `follow_up=false`.

`format: "json"` constrains Ollama to emit valid JSON, so we don't need fence-stripping fallbacks. Sampling params live inside `options` — `temperature` at the top level is silently ignored by `/api/generate`. We don't pass `num_ctx`: the model is loaded with the 262144 baked into the `-long` Modelfile, divided across `OLLAMA_NUM_PARALLEL` slots. Overriding per-request would spawn a second runner and double VRAM.

In [ ]:
PROMPT_TEMPLATE = """Does this report recommend follow-up for THIS patient?

Follow-up = patient-specific imaging/evaluation based on findings

YES if:
- Specific directive: "Repeat CT in 6 months for nodule"
- Referral for findings: "Clinical correlation recommended"
- Template applies: Patient has osteoporosis + template says "osteoporosis patients need follow-up"

NO if:
- Normal findings: "normal", "unremarkable", "stable"
- Template doesn't apply: Patient normal + template says "osteoporosis patients need follow-up"
- Generic advice: "all patients should take calcium"
- Report describes a follow-up exam, but does not request any future action

JSON: {{"follow_up": true/false, "confidence": "high"/"low", "finding": "text", "snippet": "text"}}

- finding = the clinical finding requiring follow-up (e.g., "8 mm pulmonary nodule", "indeterminate liver lesion"). "" if follow_up=false.
- snippet = short verbatim excerpt from the report containing the recommendation. "" if follow_up=false.
Read FINDINGS first to check if templates apply.

Report:
{report_text}"""


def classify_report(row):
    try:
        prompt = PROMPT_TEMPLATE.format(report_text=row.report_text)
        resp = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": MODEL,
                "prompt": prompt,
                "stream": False,
                "format": "json",
                "options": {"temperature": 0, "num_predict": 2000},
            },
            timeout=120,
        )
        resp.raise_for_status()
        obj = json.loads(resp.json()["response"])
        confidence = str(obj.get("confidence", "low")).lower()
        if confidence not in ("high", "low"):
            confidence = "low"
        return {
            "primary_report_identifier": row.primary_report_identifier,
            "followup_detected": bool(obj.get("follow_up", False)),
            "followup_confidence": confidence,
            "followup_finding": str(obj.get("finding", "")),
            "followup_snippet": str(obj.get("snippet", "")),
            "error": None,
        }
    except Exception as e:
        return {
            "primary_report_identifier": row.primary_report_identifier,
            "followup_detected": None,
            "followup_confidence": None,
            "followup_finding": None,
            "followup_snippet": None,
            "error": str(e)[:500],
        }

In [ ]:
# Pre-load the model on Ollama and pin it resident (matches OLLAMA_KEEP_ALIVE=-1).
resp = requests.post(
    f"{OLLAMA_URL}/api/generate",
    json={"model": MODEL, "keep_alive": -1},
    timeout=300,
)
resp.raise_for_status()
print(f"{MODEL} resident on Ollama")

## Test run

Classify 100 unprocessed reports end-to-end (without writing back) so you can sanity-check the model's outputs and measure throughput before the full run.

In [ ]:
df_test = (
    spark.table(WORK_TABLE)
    .filter(F.col("followup_processed_at").isNull())
    .filter(PRIORITY_FILTER)
    .limit(100)
    .select("primary_report_identifier", "report_text")
    .toPandas()
)

if len(df_test) == 0:
    print("No unprocessed priority reports to test on.")
else:
    t0 = time.time()
    results = []
    with ThreadPoolExecutor(max_workers=TOTAL_WORKERS) as ex:
        futures = [ex.submit(classify_report, row) for _, row in df_test.iterrows()]
        for f in tqdm(as_completed(futures), total=len(futures), desc="test"):
            results.append(f.result())
    elapsed = time.time() - t0
    errs = sum(1 for r in results if r["error"])
    pos = sum(1 for r in results if r["followup_detected"])
    print(f"\n{len(results)} classified in {elapsed:.1f}s "
          f"({len(results)/elapsed:.2f} reports/sec)")
    print(f"  positives: {pos}, errors: {errs}")
    for r in results[:3]:
        print(f"  - {r}")

## Full pipeline

Loops over batches. Within each batch:
1. Pull `BATCH_SIZE` unprocessed rows (priority-filtered first; falls back to remaining when priority is exhausted).
2. Classify in parallel.
3. Append errors to `ERROR_TABLE`, MERGE successes back into `WORK_TABLE`.

Idempotent — re-running picks up wherever it left off. Stop the cell at any time.

In [ ]:
result_schema = StructType([
    StructField("primary_report_identifier", StringType(),  True),
    StructField("followup_detected",         BooleanType(), True),
    StructField("followup_confidence",       StringType(),  True),
    StructField("followup_finding",          StringType(),  True),
    StructField("followup_snippet",          StringType(),  True),
    StructField("error",                     StringType(),  True),
])


def fetch_batch(spark, work_table, batch_size, priority_filter):
    """Pull the next batch of unprocessed rows, priority-filtered first."""
    base = spark.table(work_table).filter(F.col("followup_processed_at").isNull())
    if priority_filter:
        df = (base.filter(priority_filter)
                  .orderBy(F.col("message_dt").desc())
                  .limit(batch_size)
                  .select("primary_report_identifier", "report_text")
                  .toPandas())
        if len(df) > 0:
            return df, "priority"
    df = (base.orderBy(F.col("message_dt").desc())
              .limit(batch_size)
              .select("primary_report_identifier", "report_text")
              .toPandas())
    return df, "remaining"


def merge_results(spark, work_table, df_success):
    df_success.createOrReplaceTempView("_followup_results")
    spark.sql(f"""
        MERGE INTO {work_table} AS t
        USING _followup_results AS s
        ON t.primary_report_identifier = s.primary_report_identifier
        WHEN MATCHED THEN UPDATE SET
            t.followup_detected     = s.followup_detected,
            t.followup_confidence   = s.followup_confidence,
            t.followup_finding      = s.followup_finding,
            t.followup_snippet      = s.followup_snippet,
            t.followup_processed_at = current_timestamp()
    """)
    spark.catalog.dropTempView("_followup_results")


def append_errors(spark, error_table, df_errors):
    (df_errors.select(
        F.col("primary_report_identifier"),
        F.col("error"),
        F.current_timestamp().alias("error_timestamp"),
     ).write.format("delta").mode("append").saveAsTable(error_table))


total_unprocessed = (
    spark.table(WORK_TABLE).filter(F.col("followup_processed_at").isNull()).count()
)
num_batches = (total_unprocessed + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Unprocessed: {total_unprocessed:,}  |  batches: {num_batches}  |  workers: {TOTAL_WORKERS}")

overall_start = time.time()
total_processed = 0
total_errors = 0

for batch_num in tqdm(range(num_batches), desc="batches"):
    df_batch, source = fetch_batch(spark, WORK_TABLE, BATCH_SIZE, PRIORITY_FILTER)
    if len(df_batch) == 0:
        print("All reports processed.")
        break

    batch_start = time.time()
    results = []
    with ThreadPoolExecutor(max_workers=TOTAL_WORKERS) as ex:
        futures = [ex.submit(classify_report, row) for _, row in df_batch.iterrows()]
        with tqdm(total=len(df_batch), desc=f"  batch {batch_num+1} ({source})", leave=False) as pbar:
            for f in as_completed(futures):
                results.append(f.result())
                pbar.update(1)
    batch_elapsed = time.time() - batch_start

    df_results = spark.createDataFrame(results, schema=result_schema)
    df_success = df_results.filter(F.col("error").isNull()).drop("error")
    df_err     = df_results.filter(F.col("error").isNotNull())
    success_count = df_success.count()
    error_count   = df_err.count()

    if error_count > 0:
        append_errors(spark, ERROR_TABLE, df_err)
    if success_count > 0:
        merge_results(spark, WORK_TABLE, df_success)

    total_processed += success_count
    total_errors    += error_count
    throughput = len(df_batch) / batch_elapsed
    print(f"  batch {batch_num+1}/{num_batches} [{source}]: "
          f"+{success_count} processed, +{error_count} errors, "
          f"{throughput:.2f} reports/sec")

    if (batch_num + 1) % 10 == 0:
        spark.catalog.clearCache()

elapsed = time.time() - overall_start
print(f"\nDone. processed={total_processed:,}  errors={total_errors:,}  "
      f"elapsed={elapsed/3600:.2f}h  "
      f"throughput={(total_processed+total_errors)/max(elapsed,1):.2f}/s")

## Summary

In [ ]:
df = spark.table(WORK_TABLE)
total = df.count()
processed = df.filter(F.col("followup_processed_at").isNotNull()).count()
positives = df.filter(F.col("followup_detected") == True).count()
high_conf_pos = df.filter((F.col("followup_detected") == True) & (F.col("followup_confidence") == "high")).count()

print(f"Total rows:         {total:,}")
print(f"Processed:          {processed:,}  ({100*processed/total:.1f}%)")
print(f"Unprocessed:        {total - processed:,}")
if processed:
    print(f"Follow-up rate:     {100*positives/processed:.1f}%")
if positives:
    print(f"High-confidence:    {high_conf_pos:,}  ({100*high_conf_pos/positives:.1f}% of positives)")

print("\nBy follow-up status / confidence:")
df.groupBy("followup_detected", "followup_confidence").count().orderBy(
    "followup_detected", "followup_confidence"
).show()

print("\nSample high-confidence positives:")
(df.filter((F.col("followup_detected") == True) & (F.col("followup_confidence") == "high"))
   .select("primary_report_identifier", "accession_number", "modality", "followup_finding", "followup_snippet")
   .limit(10)
   .show(truncate=80))

print("\nMost common findings (high-confidence positives):")
(df.filter((F.col("followup_detected") == True) & (F.col("followup_confidence") == "high"))
   .filter(F.col("followup_finding").isNotNull() & (F.col("followup_finding") != ""))
   .groupBy("followup_finding").count()
   .orderBy(F.col("count").desc())
   .show(15, truncate=80))

if spark.catalog.tableExists(ERROR_TABLE):
    err_total = spark.table(ERROR_TABLE).count()
    print(f"\nErrors logged in {ERROR_TABLE}: {err_total:,}")
    if err_total:
        spark.table(ERROR_TABLE).groupBy("error").count().orderBy(F.col("count").desc()).show(5, truncate=80)